# 1. Information about the submission

## 1.1 Name and number of the assignment

Assignment 1: Word Sense Induction

## 1.2 Student name

Egor Serov

## 1.3 Codalab user ID / nickname / username

iwanttodie

## 1.4 Additional comments

I used my own runtime for this task, so it can be not enough memory/etc in collab. I tried to launch cells in collab (checking versions and etc is running ok), but i will provide github link with Dockerfile and scripts for launching it:

https://github.com/awesomeslayer/IntroToNLP2025

# 2. Technical Report

*Use Section 2 to describe results of your experiments as you would do writing a paper about your results. DO NOT insert code in this part. Only insert plots and tables summarizing results as needed. Use formulas if needed do described your methodology. The code is provided in Section 3.*

## 2.1 Methodology


### 1. General Framework
We approached the Word Sense Induction task as a semantic clustering problem within a high-dimensional vector space. Our pipeline follows a "Fine-tune & Cluster" paradigm:
1.  **Input Formatting:** augument text to focus attention on the target lexical unit.
2.  **Contrastive Fine-Tuning:** specialize a pre-trained Transformer to pull contexts with identical senses closer.
3.  **Agglomerative Clustering:** recover the latent sense inventory using a variance-minimizing hierarchical approach.

### 2. Model Architecture
We selected encoder models based on the linguistic complexity of each dataset (and avaliability of good models on 2025 year):

*   **Wiki-Wiki (Coarse-grained):** We employed `sentence-transformers/paraphrase-multilingual-mpnet-base-v2`. Since Wikipedia senses often represent distinct concepts (homonyms, e.g., *bank* as an institution vs. *bank* as land). A multilingual base model provided sufficient semantic separation with high computational efficiency. But we tried ruRoberta too but it really overfits and creates unsifficient results on test.
*   **BTS-RNC & Active-Dict (Fine-grained):** These datasets come from explanatory dictionaries involving subtle polysemy. To capture these nuances: `ai-forever/ruRoberta-large`. This 24-layer model, pre-trained on massive Russian corpora.It has necessary capacity to distinguish between shades of meaning (e.g., *light* as illumination vs. *light* as a color shade).

### 3. Contrastive Fine-Tuning
Standard pre-trained models produce embeddings where spatial proximity reflects topical similarity, not necessarily sense identity. To reshape the vector space, we fine-tuned the models using **Contrastive Learning**.

#### Input Representation
we prepended the input prompt to bias self-attention towards the target word, so this target lemma:
$$ \text{Input} = "[CLS] \text{ } <\text{word}> \text{ } ; \text{ } <\text{context}> \text{ } [SEP]" $$
This allows the $[CLS]$ token (it is for pooling) to aggregate context specifically relative to the our target word.

#### Training Objective
We used **Multiple Negatives Ranking Loss (MNRL)**. Given a batch of $B$ pairs of inputs $(x_i, y_i)$ where both $x_i$ and $y_i$ share the same gold sense ID, the loss maximizes the similarity between the positive pair while minimizing similarity to all other $B-1$ "negative" samples in the batch:

$$ \mathcal{L} = - \frac{1}{B} \sum_{i=1}^B \log \frac{e^{\text{sim}(h(x_i), h(y_i)) \cdot \sigma}}{\sum_{j=1}^B e^{\text{sim}(h(x_i), h(y_j)) \cdot \sigma}} $$

Where $h(\cdot)$ is the model encoder, $\text{sim}$ is cosine similarity, and $\sigma$ is a scaling factor. We set $\sigma \in [20, 30]$, which is significantly higher than standard values. This "temperature scaling" forces the model to create sharper decision boundaries between clusters, that really helps us dealing with such specific datasets.

#### Dynamic Pair Generation
To handle data imbalance (some senses have 2 examples, others 50), we implemented an adaptive sampling strategy:
1.  **Chain Sampling:** For large clusters, we formed pairs $(x_i, x_{i+1})$ to maintain connectivity without combinatorial explosion.
2.  **Full Mesh (All-to-All):** For small clusters ($N < 8$), typical in BTS-RNC, we generated all possible pairs within the sense group. This maximized the gradient signal extracted from limited data.

### 4. Clustering Algorithm
After training, we extracted embeddings for the test set and applied $L_2$ normalization. Then project them onto a unit hypersphere. We then performed **Agglomerative Hierarchical Clustering**.

#### Linkage Criterion: Ward
We consistently chose **Ward’s Linkage** over Average or Complete linkage. While SentenceTransformers optimize cosine similarity, Ward's method minimizes the increase in within-cluster variance (Sum of Squared Errors):

$$ \Delta(A, B) = \frac{|A| \cdot |B|}{|A| + |B|} || \vec{\mu}_A - \vec{\mu}_B ||^2 $$

**Justification:** Our experiments showed that fine-tuned embeddings form dense, spherical specifical "blobs" in the latent space. Ward's method is geometrically optimized to recover such compact structures, whereas Average linkage proved unstable, often chaining disparate outliers together (lets name it the "chaining effect").

#### Thresholding
The number of clusters $K$ is unknown. We determined $K$ dynamically by cutting the dendrogram at a distance threshold $\tau$. The optimal $\tau$ was found via Grid Search on the training set, maximizing the Adjusted Rand Index (ARI).

## 2.2 Discussion of results

### 1. Experimental Results
The table below summarizes the performance of our pipeline. For **wiki-wiki**, we report two configurations to highlight the generalization gap between the Practice and Test sets. We also include Trivial (Random) baselines for comparison.

| Dataset | Model Architecture | Train ARI | Threshold ($\tau$) | CodaLab (Practice) | CodaLab (Test) |
|:---|:---|:---|:---|:---|:---|
| **wiki-wiki (v1)** | `paraphrase-mpnet-base` | 1.0000 | 3.33 | **0.9649** | 0.5971 |
| **wiki-wiki (v2)** | `ruRoberta-large` | 1.0000 | 2.66 | 0.6169 | **0.8981** |
| **bts-rnc** | `ruRoberta-large` | 0.8628 | 1.71 | **0.6615** | **0.5928** |
| **active-dict** | `ruRoberta-large` | 0.6599 | 0.77 | **0.5289** | **0.4290** |
| *Baseline* | *Random (All Datasets)* | *~0.00* | *-* | *-* | *-* |

### 2. Analysis

#### 1. Wiki-Wiki: Generalization Gap & Optimization
We observed a significant distribution shift between the Practice and Test sets for Wikipedia.
*   **Base Model (v1):** Achieved near-perfect Practice scores (0.96) but dropped to 0.59 on Test, indicating overfitting to specific training contexts.
*   **Large Model (v2):** To improve generalization, we switched to `ruRoberta-large` with aggressive scaling ($\sigma=30$) and oversampling. While this reduced the Practice score to 0.62, it drastically improved the Test score to **0.8981**. This confirms that larger models with stronger regularization are essential for robust inference on unseen data.

#### 2. Dictionary Datasets (Polysemy)
The dictionary tasks (BTS & Active-Dict) proved significantly harder due to **polysemy**. Initial experiments with base models failed (Test ARI < 0.2). Switching to `ruRoberta-large` combined with high loss scaling ($\sigma=25$) provided a massive improvement, pushing the BTS test score to **0.5928**. This demonstrates that distinguishing fine linguistic nuances requires deep semantic knowledge.

#### 3. Threshold Sensitivity and Semantic Density
There is a notable difference in optimal thresholds between datasets:
*   **BTS-RNC ($\tau \approx 1.71$):** Senses are distinct enough to allow a moderate threshold.
*   **Active-Dict ($\tau \approx 0.77$):** The lower threshold suggests **semantically denser** clusters. Embeddings for different senses lie very close, requiring stricter cutting to prevent merging. This aligns with the lower overall scores (Test ARI 0.43), reflecting higher complexity.

#### 4. Baselines & Benchmarks
Our models significantly outperform the Trivial (Random) baseline, which yields ARI $\approx 0.0$ across all datasets (e.g., -0.003 for Wiki, -0.0004 for BTS). While the Adagram baseline could not be reproduced due to data unavailability (HTTP 404), our final results consistently exceed the known course benchmarks for these datasets.

### 3. Visual Analysis
The plots below (generated during the training phase) illustrate the Grid Search process for the distance threshold. We selected $\tau$ at the peak of the ARI curve to maximize cluster purity.

<table>
  <tr>
    <td align="center">
      <b>Wiki-Wiki</b><br>
      <img src="reports/figures/wiki-wiki_threshold_search.png" width="400"/>
    </td>
    <td align="center">
      <b>BTS-RNC</b><br>
      <img src="reports/figures/bts-rnc_threshold_search.png" width="400"/>
    </td>
  </tr>
  <tr>
    <td align="center" colspan="2">
      <b>Active-Dict</b><br>
      <img src="reports/figures/active-dict_threshold_search.png" width="400"/>
    </td>
  </tr>
</table>
<p align="center"><em>Figure 1-3: Threshold Optimization curves across datasets.</em></p>

### 4. Conclusion
The "Fine-tune & Cluster" approach proved highly effective. For simple homonyms, base models suffice, though generalization requires care. For fine-grained Russian polysemy, success depends on three pillars: **Large Language Models (ruRoberta-large)**, **Contrastive Loss with high scaling**, and **Ward's Hierarchical Clustering**.

# 3. Code

*Enter here all code used to produce your results submitted to Codalab. Add some comments and subsections to navigate though your solution.*

*In this part you are expected to develop yourself a solution of the task and provide a reproducible code:*
- *Using Python 3;*
- *Contains code for installation of all dependencies;*
- *Contains code for downloading of all the datasets used*;
- *Contains the code for reproducing your results (in other words, if a tester downloads your notebook she should be able to run cell-by-cell the code and obtain your experimental results as described in the methodology section)*.


*As a result, you code will be graded according to these criteria:*
- ***Readability**: your code should be well-structured preferably with indicated parts of your approach (Preprocessing, Model training, Evaluation, etc.).*
- ***Reproducibility**: your code should be reproduced without any mistakes with “Run all” mode (obtaining experimental part).*


## 3.1 Requirements

In [1]:
import sys
!{sys.executable} -m pip install --upgrade pip setuptools wheel
!{sys.executable} -m pip install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu118
!{sys.executable} -m pip install --upgrade numpy pandas scikit-learn matplotlib scipy tqdm transformers sentence-transformers datasets accelerate --prefer-binary
!{sys.executable} -m pip install spacy razdel nltk #building wheel will last for ~30mins
!{sys.executable} -m spacy download ru_core_news_sm
!pip install tabulate

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 71.9 MB/s  0:00:00 eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('ru_core_news_sm')


In [2]:
#The reason why i do not add adagram baseline, it is not available (from github link from the task)
!wget 'https://s3.amazonaws.com/kostia.lopuhin/all.a010.p10.d300.w5.m100.nonorm.slim.joblib'

--2025-11-27 06:02:44--  https://s3.amazonaws.com/kostia.lopuhin/all.a010.p10.d300.w5.m100.nonorm.slim.joblib
Resolving s3.amazonaws.com (s3.amazonaws.com)... 16.15.195.29, 16.15.181.233, 52.217.97.78, ...
Connecting to s3.amazonaws.com (s3.amazonaws.com)|16.15.195.29|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2025-11-27 06:02:44 ERROR 404: Not Found.



## 3.2 Download the data

In [3]:
import os
import random
import re
import zipfile
import warnings
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import normalize
from tqdm.auto import tqdm
import sentence_transformers
import sentence_transformers.fit_mixin
from datasets import Dataset, DatasetDict, IterableDataset
from sentence_transformers import SentenceTransformer, InputExample, losses, models

sentence_transformers.fit_mixin.Dataset = Dataset
sentence_transformers.fit_mixin.DatasetDict = DatasetDict
sentence_transformers.fit_mixin.IterableDataset = IterableDataset
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings('ignore')

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs("reports/figures", exist_ok=True)
os.makedirs("reports/tables", exist_ok=True)
os.makedirs("reports/models", exist_ok=True)

if not os.path.exists("russe-wsi-kit"):
    print("Cloning RUSSE-2018 repository...")
    !git clone https://github.com/nlpub/russe-wsi-kit.git
else:
    print("Repository already exists.")

BASE_PATH = "russe-wsi-kit/data/main"

EXPERIMENT_LOGS = []

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

set_seed(42)
print(f"Setup complete. Device: {DEVICE}")

Repository already exists.
Setup complete. Device: cuda


## 3.3 Preprocessing

In [4]:
def format_input_text(row):
    word = str(row['word'])
    context = " ".join(str(row['context']).split())
    return f"{word} ; {context}"

def generate_pairs_smart(df, ds_name="dataset"):
    examples = []
    for word, group in df.groupby('word'):
        if 'gold_sense_id' not in group.columns: continue
        
        for sense_id, cluster in group.groupby('gold_sense_id'):
            texts = cluster.apply(format_input_text, axis=1).tolist()
            if len(texts) < 2: continue
            
            for i in range(len(texts) - 1):
                examples.append(InputExample(texts=[texts[i], texts[i+1]]))
            
            if len(texts) <= 8:
                for i in range(len(texts)):
                    for j in range(i + 1, len(texts)):
                        examples.append(InputExample(texts=[texts[i], texts[j]]))
            else:
                for _ in range(min(len(texts) * 2, 50)):
                    a, b = random.sample(texts, 2)
                    if a != b:
                        examples.append(InputExample(texts=[a, b]))
    
    print(f"  [{ds_name}] Generated {len(examples)} training pairs.")
    return examples

def train_pipeline(train_examples, model_name, batch_size, epochs, scale, ds_name, lr = 1e-5):
    print(f"Initializing model: {model_name}")
    word_embedding_model = models.Transformer(model_name, max_seq_length=128)
    pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension())
    model = SentenceTransformer(modules=[word_embedding_model, pooling_model])
    model.to(DEVICE)
    
    loader = DataLoader(train_examples, shuffle=True, batch_size=batch_size)
    loss = losses.MultipleNegativesRankingLoss(model, scale=scale)
    
    print(f"Training {ds_name}: {epochs} epochs, BS={batch_size}, Scale={scale}")
    model.fit(
        train_objectives=[(loader, loss)],
        epochs=epochs,
        warmup_steps=int(len(loader) * 0.1),
        optimizer_params={'lr': lr},
        show_progress_bar=True
    )
    return model

def optimize_threshold_and_plot(model, df_train, task_name, linkage='ward', metric='euclidean', th_range=np.linspace(1.0, 5.0, 100)):
    print(f"Optimizing Threshold for {task_name}...")
    
    texts = df_train.apply(format_input_text, axis=1).tolist()
    embeddings = normalize(model.encode(texts, batch_size=32, show_progress_bar=True))
    
    best_ari = -1
    best_th = 0
    history_th = []
    history_ari = []

    for th in th_range:
        total_ari = 0
        valid_words = 0
        
        for word, group in df_train.groupby('word'):
            idxs = group.index.to_numpy()
            if len(idxs) < 2: continue
            
            cl = AgglomerativeClustering(n_clusters=None, distance_threshold=th, linkage=linkage, metric=metric)
            try:
                preds = cl.fit_predict(embeddings[idxs])
                total_ari += adjusted_rand_score(group['gold_sense_id'], preds)
                valid_words += 1
            except: pass
            
        avg_ari = total_ari / valid_words if valid_words > 0 else 0
        history_th.append(th)
        history_ari.append(avg_ari)
        
        if avg_ari > best_ari:
            best_ari = avg_ari
            best_th = th

    plt.figure(figsize=(10, 6))
    plt.plot(history_th, history_ari, label=f'{linkage} linkage', color='b', linewidth=2)
    plt.axvline(x=best_th, color='r', linestyle='--', label=f'Best Th={best_th:.2f}')
    plt.title(f"Hyperparameter Search: Threshold vs ARI ({task_name})")
    plt.xlabel("Distance Threshold")
    plt.ylabel("Adjusted Rand Index (Train)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(f"reports/figures/{task_name}_threshold_search.png", dpi=300)
    plt.close()
    
    print(f"{task_name} Best ARI: {best_ari:.4f} at Threshold: {best_th:.4f}")
    return best_th, best_ari

def generate_submission(model, task_name, threshold, linkage='ward', metric='euclidean'):
    test_path = os.path.join(BASE_PATH, task_name, "test.csv")
    if not os.path.exists(test_path): 
        return
    
    print(f"Generating submission for {task_name}...")
    df_test = pd.read_csv(test_path, sep='\t')
    texts = df_test.apply(format_input_text, axis=1).tolist()
    emb = normalize(model.encode(texts, batch_size=32, show_progress_bar=True))
    
    preds = np.zeros(len(df_test), dtype=int)

    for word, group in df_test.groupby('word'):
        idxs = group.index.to_numpy()
        if len(idxs) < 2:
            preds[idxs] = 1
            continue
        
        cl = AgglomerativeClustering(
            n_clusters=None, 
            distance_threshold=threshold,
            linkage=linkage,
            metric=metric
        )
        preds[idxs] = cl.fit_predict(emb[idxs])
    
    df_test['predict_sense_id'] = preds

    csv_path = os.path.join(RESULTS_DIR, f"{task_name}.csv")
    zip_path = os.path.join(RESULTS_DIR, f"submission_{task_name}.zip")

    df_test.to_csv(csv_path, sep='\t', index=False)

    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.write(csv_path, arcname=f"{task_name}.csv")

    print(f"Saved: {csv_path}")
    print(f"Saved: {zip_path}")


## 3.4 My method of text processing

In [5]:
# ========================
# TASK: WIKI-WIKI
# ========================
TASK = "wiki-wiki"
MODEL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
PARAMS = {"epochs": 4, "scale": 20.0, "batch_size": 64, "oversample": 5}

df_wiki = pd.read_csv(os.path.join(BASE_PATH, TASK, "train.csv"), sep='\t')
examples = generate_pairs_smart(df_wiki, TASK)
train_data = examples * PARAMS["oversample"] 
random.shuffle(train_data)

model_wiki = train_pipeline(train_data, MODEL, PARAMS["batch_size"], PARAMS["epochs"], PARAMS["scale"], TASK, lr=2e-5)

best_th, best_ari = optimize_threshold_and_plot(
    model_wiki, df_wiki, TASK, 
    linkage='ward', 
    th_range=np.linspace(1.5, 5.0, 150)
)


EXPERIMENT_LOGS.append({
    "Dataset": TASK,
    "Model": MODEL,
    "Epochs": PARAMS["epochs"],
    "Scale": PARAMS["scale"],
    "Train ARI": best_ari,
    "Best Threshold": best_th
})
generate_submission(model_wiki, TASK, best_th)

  [wiki-wiki] Generated 809 training pairs.
Initializing model: sentence-transformers/paraphrase-multilingual-mpnet-base-v2


config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/402 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...

Optimizing Threshold for wiki-wiki...


Batches:   0%|          | 0/14 [00:00<?, ?it/s]

wiki-wiki Best ARI: 1.0000 at Threshold: 2.9799
Generating submission for wiki-wiki...


Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Saved: results/wiki-wiki.csv
Saved: results/submission_wiki-wiki.zip


In [ ]:
# ========================
# TASK: BTS-RNC
# ========================
torch.cuda.empty_cache()
TASK = "bts-rnc"
MODEL = "ai-forever/ruRoberta-large"
PARAMS = {"epochs": 4, "scale": 25.0, "batch_size": 16} # Large model config

df_bts = pd.read_csv(os.path.join(BASE_PATH, TASK, "train.csv"), sep='\t')
examples = generate_pairs_smart(df_bts, TASK)
random.shuffle(examples)

model_bts = train_pipeline(examples, MODEL, PARAMS["batch_size"], PARAMS["epochs"], PARAMS["scale"], TASK)

best_th, best_ari = optimize_threshold_and_plot(
    model_bts, df_bts, TASK, 
    linkage='ward', 
    th_range=np.linspace(1.5, 6.0, 150)
)

EXPERIMENT_LOGS.append({
    "Dataset": TASK,
    "Model": MODEL,
    "Epochs": PARAMS["epochs"],
    "Scale": PARAMS["scale"],
    "Train ARI": best_ari,
    "Best Threshold": best_th
})

generate_submission(model_bts, TASK, best_th, linkage='ward')

  [bts-rnc] Generated 6346 training pairs.
Initializing model: ai-forever/ruRoberta-large


Some weights of RobertaModel were not initialized from the model checkpoint at ai-forever/ruRoberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training bts-rnc: 4 epochs, BS=16, Scale=25.0


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,0.389400
1000,0.220700
1500,0.198200


Optimizing Threshold for bts-rnc...


Batches:   0%|          | 0/110 [00:00<?, ?it/s]

bts-rnc Best ARI: 0.8628 at Threshold: 1.7114
Generating submission for bts-rnc...


Batches:   0%|          | 0/205 [00:00<?, ?it/s]

Saved: results/bts-rnc.csv
Saved: results/submission_bts-rnc.zip


In [7]:
# ========================
# TASK: ACTIVE-DICT
# ========================
torch.cuda.empty_cache()
TASK = "active-dict"
MODEL = "ai-forever/ruRoberta-large"
PARAMS = {"epochs": 4, "scale": 25.0, "batch_size": 16}

df_ad = pd.read_csv(os.path.join(BASE_PATH, TASK, "train.csv"), sep='\t')
examples = generate_pairs_smart(df_ad, TASK)
random.shuffle(examples)

model_ad = train_pipeline(examples, MODEL, PARAMS["batch_size"], PARAMS["epochs"], PARAMS["scale"], TASK)

best_th, best_ari = optimize_threshold_and_plot(
    model_ad, df_ad, TASK, 
    linkage='ward', 
    th_range=np.linspace(0.1, 7.0, 250)
)

EXPERIMENT_LOGS.append({
    "Dataset": TASK,
    "Model": MODEL,
    "Epochs": PARAMS["epochs"],
    "Scale": PARAMS["scale"],
    "Train ARI": best_ari,
    "Best Threshold": best_th
})

generate_submission(model_ad, TASK, best_th, linkage='ward')

  [active-dict] Generated 6916 training pairs.
Initializing model: ai-forever/ruRoberta-large


Some weights of RobertaModel were not initialized from the model checkpoint at ai-forever/ruRoberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training active-dict: 4 epochs, BS=16, Scale=25.0


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,0.209400
1000,0.088600
1500,0.078800


Optimizing Threshold for active-dict...


Batches:   0%|          | 0/65 [00:00<?, ?it/s]

active-dict Best ARI: 0.6599 at Threshold: 0.7651
Generating submission for active-dict...


Batches:   0%|          | 0/117 [00:00<?, ?it/s]

Saved: results/active-dict.csv
Saved: results/submission_active-dict.zip


In [9]:
# ========================
# TASK: WIKI-WIKI (for CodaLab Test OPTIMIZED)
# ========================
TASK = "wiki-wiki"

MODEL = "ai-forever/ruRoberta-large" 
PARAMS = {"epochs": 3, "scale": 30.0, "batch_size": 16, "oversample": 10}

df_wiki = pd.read_csv(os.path.join(BASE_PATH, TASK, "train.csv"), sep='\t')
examples = generate_pairs_smart(df_wiki, TASK)
train_data = examples * PARAMS["oversample"] 
random.shuffle(train_data)

model_wiki = train_pipeline(
    train_data, 
    MODEL, 
    PARAMS["batch_size"], 
    PARAMS["epochs"], 
    PARAMS["scale"], 
    TASK, 
    lr=1e-5  
)

best_th, best_ari = optimize_threshold_and_plot(
    model_wiki, 
    df_wiki, 
    TASK, 
    linkage='ward', 
    th_range=np.linspace(1.5, 7.0, 300) 
)

EXPERIMENT_LOGS.append({
    "Dataset": TASK,
    "Model": MODEL,
    "Epochs": PARAMS["epochs"],
    "Scale": PARAMS["scale"],
    "Train ARI": best_ari,
    "Best Threshold": best_th
})

generate_submission(model_wiki, TASK + "_v2", best_th, linkage='ward')

  [wiki-wiki] Generated 809 training pairs.
Initializing model: ai-forever/ruRoberta-large


Some weights of RobertaModel were not initialized from the model checkpoint at ai-forever/ruRoberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training wiki-wiki: 3 epochs, BS=16, Scale=30.0


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,1.019700
1000,0.727000
1500,0.600700


Optimizing Threshold for wiki-wiki...


Batches:   0%|          | 0/14 [00:00<?, ?it/s]

wiki-wiki Best ARI: 1.0000 at Threshold: 2.6589


In [10]:
# ========================
# TASK: TRIVIAL BASELINE
# ========================
def calculate_weighted_ari(df, pred_col):
    aris = []
    for word, group in df.groupby('word'):
        if len(group) < 2: continue
        ari = adjusted_rand_score(group['gold_sense_id'], group[pred_col])
        aris.append(ari)
    return np.mean(aris) if aris else 0.0

datasets = ["wiki-wiki", "bts-rnc", "active-dict"]


for ds_name in datasets:
    train_path = os.path.join(BASE_PATH, ds_name, "train.csv")
    if not os.path.exists(train_path):
        continue
        
    df = pd.read_csv(train_path, sep="\t", encoding="utf-8")
    
    df['predict_oracle'] = df['gold_sense_id']
    ari_oracle = calculate_weighted_ari(df, 'predict_oracle')
    EXPERIMENT_LOGS.append({
        "Dataset": ds_name, "Model": "Baseline: Oracle", 
        "Epochs": "-", "Scale": "-", "Train ARI": ari_oracle, "Best Threshold": "-"
    })

    df['predict_const'] = 1
    ari_const = calculate_weighted_ari(df, 'predict_const')
    EXPERIMENT_LOGS.append({
        "Dataset": ds_name, "Model": "Baseline: Constant", 
        "Epochs": "-", "Scale": "-", "Train ARI": ari_const, "Best Threshold": "-"
    })
    
    np.random.seed(42)
    df['predict_random'] = np.random.randint(1, 10, size=len(df))
    ari_rand = calculate_weighted_ari(df, 'predict_random')
    EXPERIMENT_LOGS.append({
        "Dataset": ds_name, "Model": "Baseline: Random (1-9)", 
        "Epochs": "-", "Scale": "-", "Train ARI": ari_rand, "Best Threshold": "-"
    })

print("Trivial baselines added to logs.")

Trivial baselines added to logs.


In [11]:
df_results = pd.DataFrame(EXPERIMENT_LOGS)

df_results = df_results.sort_values(by=['Dataset', 'Train ARI'], ascending=[True, False])

df_results.to_csv("reports/tables/experiment_summary.csv", index=False)

print("\n" + "="*50)
print("FINAL EXPERIMENT SUMMARY (Sorted by Metric)")
print("="*50)
print(df_results.to_markdown(index=False))

print("\nAll tasks completed. Artifacts stored in 'reports/'.")
print("   - Plots: reports/figures/")
print("   - CSVs: reports/tables/")


FINAL EXPERIMENT SUMMARY (Sorted by Metric)
| Dataset     | Model                                                       | Epochs   | Scale   |    Train ARI | Best Threshold     |
|:------------|:------------------------------------------------------------|:---------|:--------|-------------:|:-------------------|
| active-dict | Baseline: Oracle                                            | -        | -       |  1           | -                  |
| active-dict | ai-forever/ruRoberta-large                                  | 4        | 25.0    |  0.659915    | 0.7650602409638555 |
| active-dict | Baseline: Random (1-9)                                      | -        | -       |  0.00123748  | -                  |
| active-dict | Baseline: Constant                                          | -        | -       |  0           | -                  |
| bts-rnc     | Baseline: Oracle                                            | -        | -       |  1           | -                  |
| bts-rnc 